### Checking the performance of CSV v/s parquet formats

In [23]:
import pandas as pd
from pathlib import Path

DATA_PATH_CSV = Path("../data/processed/index/master/nifty50_index_master_2016_2026.csv")

df = pd.read_csv(DATA_PATH_CSV)

df

,index_name,open_index_value,high_index_value,low_index_value,closing_index_value,points_change,change(%),volume,turnover_(rs._cr.),p/e,p/b,div_yield,trade_date
0,NIFTY 50,7938.45,7972.55,7909.80,7963.20,16.85,0.21,64843836,2613.91,21.53,3.19,1.45,2016-01-01
1,NIFTY 50,7589.50,7600.45,7541.25,7555.95,-7.60,-0.10,206645021,8209.50,20.22,3.03,1.53,2016-02-01
2,NIFTY 50,7038.25,7235.50,7035.10,7222.30,235.25,3.37,283561570,9896.58,19.53,2.90,1.60,2016-03-01
3,NIFTY 50,7718.05,7740.15,7666.10,7713.05,-25.35,-0.33,189571551,8118.47,21.19,3.26,1.45,2016-04-01
4,NIFTY 50,8179.20,8215.35,8171.05,8179.95,19.85,0.24,200371829,8408.53,22.65,3.41,1.31,2016-06-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2514,NIFTY 50,25063.35,25246.65,24932.55,25175.40,126.75,0.51,618696080,50989.25,21.91,3.42,1.33,2026-01-27
2515,NIFTY 50,25459.85,25476.40,25141.30,25178.65,-317.90,-1.25,438924282,38634.68,22.03,3.42,1.24,2026-02-27
2516,NIFTY 50,25258.85,25372.10,25187.65,25342.75,167.35,0.66,574903942,42364.52,22.07,3.45,1.30,2026-01-28
2517,NIFTY 50,25345.00,25458.15,25159.80,25418.90,76.15,0.30,582389331,44306.95,22.12,3.46,1.30,2026-01-29


In [ ]:
import pandas as pd
from pathlib import Path

DATA_PATH = Path("../data/processed/index/parquet/nifty50_index_master_2016_2026.parquet")
df = pd.read_parquet(DATA_PATH)

df.head()

<b>Note:</b><i>From above example we can't see the difference between csv & parquet. But, <b>Parquet is more faster than csv</b> (we can see the clear difference in data of larger files. for ex. equity master dataset (having rows 4.2+ M))</i>

## Daily Return

In [ ]:
df["return_1d"] = df["closing_index_value"].pct_change()
df.head()

pct_change() calculates the percentage change between the current value and the previous value.</br>
Return=(Current Price−Previous Price​)/Previous Price</br>
If the Result is 0.05 then the returns are 0.05*100 = 5% High from last closing

## Weekly Momentum

In [ ]:
df["return_5d"] = df["closing_index_value"].pct_change(5)
df

## Monthly Momentum

In [ ]:
df["return_20d"] = df["closing_index_value"].pct_change(20)
df

## Rolling Volatility

In [ ]:
df["volatility_20d"] = df["return_1d"].rolling(20).std()
df

moving window of 20 rows (20 days)</br>
looks at the previous 20 days of returns_1d</br>
std() calculates the standard deviation of the values inside the window</br>
In simple words, How much the market has been fluctuating over the last 20 days</br>
Higher std → more volatile market</br>
Lower std → stable market</br>
| Volatility (`volatility_20d`) | Approx %    | Market Type | Meaning                          |
| ----------------------------- | ----------- | ----------- | -------------------------------- |
| **< 0.007**                   | < 0.7%      | Low         | Stable / trending market         |
| **0.007 – 0.015**             | 0.7% – 1.5% | Medium      | Normal market movement           |
| **> 0.015**                   | > 1.5%      | High        | Panic / crash / highly uncertain |


## Drawdown

In [ ]:
rolling_max = df["closing_index_value"].cummax()
df["drawdown"] = df["closing_index_value"] / rolling_max - 1
df

cummax() means cumulative maximum</br>
(highest value the index has reached so far)</br>
Drawdown = (Current Price​)/(Peak Price) - 1</br>
how far the price has fallen from its highest point</br>
If drawdown is -0.028 then -2.8% drawdown (Meaning price is 2.8% below its peak)

## Volume Momentum

In [ ]:
df["volume_change"] = df["volume"].pct_change(5)
df.head(6)

## Save Features (Partitioned Parquet)

In [ ]:
df["year"] = df["trade_date"].dt.year

In [ ]:
output_path = Path("../data/features/index")
output_path.mkdir(parents=True, exist_ok=True)

years = df["year"].unique()

for year in years:
    
    df_year = df[df["year"] == year]
    
    file_path = output_path / f"index_features_{year}.parquet"
    
    if not file_path.exists():
        df_year.to_parquet(file_path, index=False)